In [1]:
# Se importa solo los modulos relacionados con el ETL- Extract, Transform and Load
import numpy as np
import pandas as pd
import ast

In [2]:
# Se extraen los dos archivos
df_credits= pd.read_csv(
    r"c:\Users\param\OneDrive\Escritorio\Programacion\Soy Henry\Proyecto individual 1\DATASETS\credits.csv",dtype={10: str})


df_movies = pd.read_csv(
    r"c:\Users\param\OneDrive\Escritorio\Programacion\Soy Henry\Proyecto individual 1\DATASETS\movies_dataset.csv",dtype={10: str})

In [3]:
# Se juntan las dos tablas al inicio para evitar errores de concatenacion posterior
data = pd.merge(df_movies, df_credits, left_index=True, right_index=True)
# Se elimina todas las columnas que no se van a utilizar
data.drop(["video", "imdb_id", "adult", "original_title", "homepage", "poster_path", "status", "spoken_languages"], axis=1, inplace=True) 
data=pd.DataFrame(data)

Eliminamos todos los registros nulos de las variables mas importantes- como hay restriccion de espacio no se trataran los valor nulos.

In [4]:
data = data.dropna(subset=["release_date"])
data = data.dropna(subset=["vote_average"])
data = data.dropna(subset=["vote_count"])
data=data.dropna(subset=["belongs_to_collection"])
data=data.dropna(subset=["cast"])
data=data.dropna(subset=["production_companies"])

1.1  Se inicia encontrando la variable actor principal- sólo se toma el actor principal que se encuentra en el primer diccionario de la variable "cast". Esta decisión corresponde a una medida de rendimiento.

In [5]:
data["cast"] = data["cast"].apply(ast.literal_eval)
data["first_cast_member"] = data["cast"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
data["nombre_actor"] = data["first_cast_member"].apply(lambda x: x["name"] if isinstance(x, dict) and "name" in x else None)
data.drop(["first_cast_member","id_y","cast"],axis=1,inplace=True)

1.2 Se toma la variable "crew". En esta el director esta en diccionarios distintos por lo que requiere una busqueda mas exhaustiva.

In [6]:
# Creamos un array job:director, name:nombre_director
df_credits["crew"] = df_credits["crew"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else x)
df_director = df_credits["crew"].apply(
    lambda x: next(
        ({"trabajo": d["job"], "nombre": d["name"]} for d in x if d["job"] == "Director"),
        None
    ) if isinstance(x, list) else None
)


In [7]:
#Solo dejamos el nombre del director y eliminamos el resto
df_director["nombre_dir"]=df_director.apply(lambda x: list(x.values())[1] if isinstance(x, dict) else None)
#Concatenamos este nombre con la base de datos que esta juntada y eliminamos crew
data = pd.concat([data, df_director["nombre_dir"].rename("nombre_director")], axis=1)
#Eliminamos la columna crew y id_crew
data.drop(["crew"],axis=1,inplace=True)

1.3 Desanidamos "Belongs_to_collection" para obtener el nombre de la pelicula

In [8]:
data=data.dropna(subset=["belongs_to_collection"])
data["belongs_to_collection"] = data["belongs_to_collection"].apply(ast.literal_eval)
data["titulo_pelicula"] = data["belongs_to_collection"].apply(lambda x: x["name"] if isinstance(x, dict) and "name" in x else None)
data.drop(["belongs_to_collection"],axis=1, inplace=True)

1.4 Desanidamos "production companies"

In [9]:
data=pd.DataFrame(data)
data=data.dropna(subset=["production_companies"])
data['production_companies'] = data['production_companies'].apply(ast.literal_eval)
productora= data['production_companies'].apply(lambda x: x[0]['name'] if isinstance(x, list) and len(x) > 0 else None)
data = pd.concat([data, productora.rename("nombre_productora")], axis=1)
data.drop(["production_companies"],axis=1, inplace=True)
data.drop(["id_x"],axis=1, inplace=True)


1.5 Desanidamos genero- importante para el modelo

In [10]:
data=pd.DataFrame(data)
data=data.dropna(subset=["genres"])
data['genres'] = data['genres'].apply(ast.literal_eval)
genero= data['genres'].apply(lambda x: x[0]['name'] if isinstance(x, list) and len(x) > 0 else None)
data = pd.concat([data, genero.rename("genero")], axis=1)
data.drop(["genres"],axis=1, inplace=True)

1.6 Desanidamos pais productor- es relevante para el modelo

In [11]:
data=pd.DataFrame(data)
data=data.dropna(subset=["production_countries"])
data['production_countries'] = data['production_countries'].apply(ast.literal_eval)
pais= data['production_countries'].apply(lambda x: x[0]['name'] if isinstance(x, list) and len(x) > 0 else None)
data = pd.concat([data, pais.rename("pais_origen")], axis=1)
data.drop(["production_countries"],axis=1, inplace=True)


2.1 Relleno de datos nulos por 0- como lo establece las indicaciones

In [12]:
data['revenue'] = data['revenue'].fillna(0)
data['budget'] = data['budget'].fillna(0)

3.1 Manejo de la variable release date para crear las variables año, mes y dia de estreno

In [13]:
# Se convierte el tipo object a date time
data['release_date'] = pd.to_datetime(data['release_date'],errors="coerce")
data['release_date'] = data['release_date'].dt.strftime('%Y-%m-%d')
data['release_date'] = pd.to_datetime(data['release_date'])
#Saca el año
data['release_año'] = data['release_date'].dt.year
#Saca el mes del release con un entero
data["release_month"]=data["release_date"].dt.month
#Saca el dia de la semana como un entero- 0 es lunes y domingo es 6
data["release_day"]=data["release_date"].dt.day_of_week



In [14]:
#Creamos un mapa para pasar los enteros a "strings"
# Mapping el diccionario dia
mes_mapa = {   1: 'enero',
    2: 'febrero',
    3: 'marzo',
    4: 'abril',
    5: 'mayo',
    6: 'junio',
    7: 'julio',
    8: 'agosto',
    9: 'septiembre',
    10: 'octubre',
    11: 'noviembre',
    12: 'diciembre'
}
 

semana_mapa = {
    0: 'lunes',
    1: 'martes',
    2: 'miercoles',
    3: 'jueves',
    4: 'viernes',
    5: 'sabado',
    6: 'domingo'
}


data["release_mes"] = data['release_month'].map(mes_mapa)
data["release_dia"]=data["release_day"].map(semana_mapa)
#Elimino las columnas viejas y solo me quedo con las necesarias para crear las funciones
data=data.drop("release_month",axis=1)
data=data.drop("release_day",axis=1)
data=data.drop("release_date",axis=1)


4.1 Se crea la columna return con los campos revenue y budget

In [15]:
# Volver los valores a numeros
data["revenue"]=pd.to_numeric(data["revenue"])
data["budget"]=pd.to_numeric(data["budget"],errors="coerce")

# Pasarlos a formato float
data['revenue'] = data['revenue'].astype(float)
data['budget'] = data['budget'].astype(float)

# Crear division-nueva variable
return_per=data["revenue"]/data["budget"]
data = pd.concat([data, return_per], axis=1)
data = data.rename(columns={0: 'return'})
data['return'] = data['return'].fillna(0)



Se borran los nulos y ceros... primero se verifica

In [20]:
data = data.dropna(subset=["nombre_actor","nombre_director"])

5.1 Se crean las diferentes bases de datos para correr las diferentes funciones.

In [22]:
#Se crean diferentes tablas para que la funcion solo suba la tabla que requiera para obtener el resultado
tabla_mes=data["release_mes"]
#Nombre con el que se guardará el archivo
nombre_mes = "df_tabla_mes.csv"

#Para dia
tabla_dia=data["release_dia"]
nombre_dia="df_tabla_dia"
#Para titulo
tabla_titulo=data[["release_año","titulo_pelicula","popularity","vote_average"]]
nombre_titulo="df_tabla_titulo"
#Para voto > 2000
tabla_voto_2000 = data.loc[data["vote_count"] > 2000, ["titulo_pelicula", "vote_average", "release_año"]]
nombre_voto_2000="df_tabla_voto"
#Para actor
tabla_actor=data[["nombre_actor","return"]]
nombre_actor="df_tabla_actor"
#Para director
tabla_director=data[["nombre_director","return","titulo_pelicula","revenue","budget","release_año","release_mes","release_dia"]]
nombre_director="df_nombre_director"
#Para EDA
tabla_eda=data
eda="df_data_EDA"

file_path = r"C:\Users\param\OneDrive\Escritorio\Programacion\Soy Henry\Proyecto individual 1\DATASETS"


In [23]:
import os

full_path_1 = os.path.join(file_path, nombre_mes)
tabla_mes.to_csv(full_path_1, index=False)

full_path_2=os.path.join(file_path,nombre_dia)
tabla_dia.to_csv(full_path_2, index=False)

full_path_3=os.path.join(file_path,nombre_titulo)
tabla_titulo.to_csv(full_path_3, index=False)

full_path_4=os.path.join(file_path,nombre_voto_2000)
tabla_voto_2000.to_csv(full_path_4, index=False)

full_path_5=os.path.join(file_path,nombre_actor)
tabla_actor.to_csv(full_path_5, index=False)

full_path_6=os.path.join(file_path,nombre_director)
tabla_director.to_csv(full_path_6, index=False)

full_path_7=os.path.join(file_path,eda)
tabla_eda.to_csv(full_path_7, index=False)